#Mount Google Drive & Setup Folder

In [1]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Tentukan folder tujuan
PROJECT_FOLDER = "/content/drive/MyDrive/ai-telemed-copilot"

# 3. Buat folder jika belum ada
if not os.path.exists(PROJECT_FOLDER):
    os.makedirs(PROJECT_FOLDER)
    print(f"Folder dibuat: {PROJECT_FOLDER}")
else:
    print(f"Folder ditemukan: {PROJECT_FOLDER}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder ditemukan: /content/drive/MyDrive/ai-telemed-copilot


#Import Library & Konfigurasi

In [2]:
import json
import glob
import zipfile
import pandas as pd

# --- KONFIGURASI ---
MAX_DRUGS_TO_PROCESS = 5000
OUTPUT_FILENAME = "drug_kb.jsonl"
OUTPUT_PATH = os.path.join(PROJECT_FOLDER, OUTPUT_FILENAME)

print(f"Target Output: {OUTPUT_PATH}")

Target Output: /content/drive/MyDrive/ai-telemed-copilot/drug_kb.jsonl


#Download Dataset OpenFDA

In [3]:
import os

print("Mendownload 1 File Spesifik langsung dari OpenFDA (Official Source)...")

URL_TARGET = "https://download.open.fda.gov/drug/label/drug-label-0001-of-0012.json.zip"
ZIP_FILENAME = "drug-label-0001-of-0012.json.zip"

# Download pakai wget
!wget {URL_TARGET} -O {ZIP_FILENAME}

print(f"Download selesai: {ZIP_FILENAME}")

Mendownload 1 File Spesifik langsung dari OpenFDA (Official Source)...
--2026-02-01 05:59:20--  https://download.open.fda.gov/drug/label/drug-label-0001-of-0012.json.zip
Resolving download.open.fda.gov (download.open.fda.gov)... 3.162.163.36, 3.162.163.91, 3.162.163.17, ...
Connecting to download.open.fda.gov (download.open.fda.gov)|3.162.163.36|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 145683423 (139M) [application/zip]
Saving to: ‘drug-label-0001-of-0012.json.zip’

drug-label-0001-of- 100%[===================>] 138.93M  71.0MB/s    in 2.0s    

2026-02-01 05:59:23 (71.0 MB/s) - ‘drug-label-0001-of-0012.json.zip’ saved [145683423/145683423]

Download selesai: drug-label-0001-of-0012.json.zip


#Unzip File

In [4]:
import zipfile
import glob

print("Mengekstrak file ZIP...")

# Nama file zip
target_zip = "drug-label-0001-of-0012.json.zip"

if os.path.exists(target_zip):
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(".")

        # Cek nama file hasil ekstrak
        extracted_files = zip_ref.namelist()
        print(f"Berhasil mengekstrak: {extracted_files}")
else:
    print(f"Error: File {target_zip} tidak ditemukan. Cek Cell 4.")

Mengekstrak file ZIP...
Berhasil mengekstrak: ['drug-label-0001-of-0012.json']


#Parsing & Simpan

In [5]:
import json
import glob

print("Mulai Parsing JSON dan Membuat Knowledge Base...")

processed_count = 0
results = []

# Cari file JSON hasil ekstrak
# Exclude file konfigurasi lain, fokus ke file data obat
json_files = [f for f in glob.glob("*.json") if "drug-label" in f]

if json_files:
    json_filename = json_files[0]
    print(f"Membaca file target: {json_filename}")

    with open(json_filename, 'r') as f:
        # Load JSON
        data = json.load(f)

        # Structure data ada di key 'results'
        drug_list = data.get('results', [])

        for drug in drug_list:
            if processed_count >= MAX_DRUGS_TO_PROCESS:
                break

            # --- LOGIC PARSING ---
            openfda_info = drug.get('openfda', {})
            brand_names = openfda_info.get('brand_name', [])
            generic_names = openfda_info.get('generic_name', [])

            all_names = [n.lower() for n in brand_names + generic_names]

            if not all_names:
                continue

            dosage = " ".join(drug.get('dosage_and_administration', ['Data not available']))
            warnings = " ".join(drug.get('warnings', ['Data not available']))
            contra = " ".join(drug.get('contraindications', ['Data not available']))
            interactions = " ".join(drug.get('drug_interactions', ['Data not available']))

            drug_entry = {
                "names": all_names,
                "dosage": dosage[:1000],
                "warnings": warnings[:1000],
                "contra": contra[:1000],
                "interactions": interactions[:1000]
            }

            results.append(drug_entry)
            processed_count += 1

    print(f"Berhasil memproses {len(results)} obat.")

    # Simpan ke Google Drive
    if 'OUTPUT_PATH' in locals():
        print(f"Menyimpan ke: {OUTPUT_PATH}")
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            for entry in results:
                json.dump(entry, f)
                f.write('\n')
        print("=== SUKSES! FILE TERSIMPAN DI DRIVE ===")
    else:
        print("Warning: Variable OUTPUT_PATH belum didefinisikan. Jalankan Cell 2 dulu.")

else:
    print("Error: File JSON tidak ditemukan. Pastikan Cell 5 berhasil unzip.")

Mulai Parsing JSON dan Membuat Knowledge Base...
Membaca file target: drug-label-0001-of-0012.json
Berhasil memproses 5000 obat.
Menyimpan ke: /content/drive/MyDrive/ai-telemed-copilot/drug_kb.jsonl
=== SUKSES! FILE TERSIMPAN DI DRIVE ===


#Cek Data Final

In [6]:
import os
import json

print("--- MULAI VERIFIKASI DATA ---")

if os.path.exists(OUTPUT_PATH):
    # 1. Cek Ukuran File
    file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"File ditemukan: {OUTPUT_PATH}")
    print(f"Ukuran file: {file_size_mb:.2f} MB")

    # 2. Baca 3 Baris Pertama sebagai Sampel
    print("\n--- SAMPEL DATA (3 OBAT PERTAMA) ---")

    with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 3:
                break

            data = json.loads(line)

            # Tampilkan info dasar untuk memastikan parsing benar
            print(f"Data #{i+1}")
            print(f"Nama Obat (List): {data.get('names')}")

            # Cek panjang teks untuk memastikan kita berhasil memotong teks (truncation)
            dosage_len = len(data.get('dosage', ''))
            warnings_len = len(data.get('warnings', ''))

            print(f"Panjang Teks Dosage: {dosage_len} karakter")
            print(f"Panjang Teks Warnings: {warnings_len} karakter")
            print("-" * 30)

    print("\nVerifikasi Selesai. Data siap didownload.")

else:
    print(f"ERROR: File tidak ditemukan di {OUTPUT_PATH}")
    print("Pastikan Cell sebelumnya sudah dijalankan sampai selesai.")

--- MULAI VERIFIKASI DATA ---
File ditemukan: /content/drive/MyDrive/ai-telemed-copilot/drug_kb.jsonl
Ukuran file: 7.95 MB

--- SAMPEL DATA (3 OBAT PERTAMA) ---
Data #1
Nama Obat (List): ['sterile diluent for treprostinil', 'water']
Panjang Teks Dosage: 1000 karakter
Panjang Teks Warnings: 18 karakter
------------------------------
Data #2
Nama Obat (List): ['dextroamphetamine saccharate, amphetamine aspartate monohydrate, dextroamphetamine sulfate, and amphetamine sulfate', 'dextroamphetamine saccharate, amphetamine aspartate monohydrate, dextroamphetamine sulfate, and amphetamine sulfate']
Panjang Teks Dosage: 1000 karakter
Panjang Teks Warnings: 1000 karakter
------------------------------
Data #3
Nama Obat (List): ['hydrocodone bitartrate', 'hydrocodone bitartrate']
Panjang Teks Dosage: 1000 karakter
Panjang Teks Warnings: 18 karakter
------------------------------

Verifikasi Selesai. Data siap didownload.


#Cek Obat Tertentu

In [7]:
# Daftar obat yang ingin dicek (gunakan huruf kecil)
# Catatan: Dataset ini dari US, jadi gunakan nama US (cth: Paracetamol -> Acetaminophen)
cek_list = [
    "aspirin",
    "ibuprofen",
    "acetaminophen", # Paracetamol
    "amoxicillin",
    "metformin",
    "lisinopril",
    "tylenol",       # Brand populer
    "advil",         # Brand populer
    "lipitor"        # Brand populer
]

print(f"Sedang mencari {len(cek_list)} obat umum dalam database...")

# Siapkan dictionary untuk melacak status ketemu/tidak
status_obat = {nama: False for nama in cek_list}
jumlah_ketemu = 0

if 'OUTPUT_PATH' in locals() and os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            # Baca baris data
            data = json.loads(line)
            nama_obat_di_database = data.get('names', [])

            # Cek setiap target obat
            for target in cek_list:
                # Jika target belum ketemu, cari
                if not status_obat[target]:
                    # Cek apakah target ada di dalam list nama obat database
                    # Kita pakai 'in' untuk pencarian exact match di dalam list
                    if target in nama_obat_di_database:
                        status_obat[target] = True
                        jumlah_ketemu += 1

    # Tampilkan Laporan
    print("\n--- HASIL PENCARIAN OBAT ---")
    for obat in cek_list:
        if status_obat[obat]:
            print(f"[ADA]   : {obat}")
        else:
            print(f"[ABSEN] : {obat}")

    print("-" * 30)
    print(f"Total ditemukan: {jumlah_ketemu} dari {len(cek_list)}")

    if jumlah_ketemu < 3:
        print("\nCATATAN: Hasil temu sedikit. Ini normal karena kita membatasi limit hanya 2000 data.")
        print("Jika ingin lebih lengkap, Anda bisa kembali ke Cell 2 dan menaikkan MAX_DRUGS_TO_PROCESS (misal jadi 5000 atau 10000).")
    else:
        print("\nKesimpulan: Dataset cukup representatif untuk demo.")

else:
    print("Error: Path file tidak ditemukan. Pastikan variabel OUTPUT_PATH sudah terdefinisi.")

Sedang mencari 9 obat umum dalam database...

--- HASIL PENCARIAN OBAT ---
[ADA]   : aspirin
[ADA]   : ibuprofen
[ADA]   : acetaminophen
[ADA]   : amoxicillin
[ADA]   : metformin
[ADA]   : lisinopril
[ABSEN] : tylenol
[ABSEN] : advil
[ABSEN] : lipitor
------------------------------
Total ditemukan: 6 dari 9

Kesimpulan: Dataset cukup representatif untuk demo.
